# Astropedia: fine-tune SmolLM2-135M

Runtime → Change runtime type → **T4 GPU** (free) or better. One run takes about 1–2 hours on a T4 for ~30k examples.

1. Run the cells in order. The upload cell asks for `train.jsonl` and `val.jsonl` from `ml/data/sft/`.
2. The last cell downloads `astro-135m.zip`. Unzip it into `ml/models/astro-135m/` and run the export script locally.

In [ ]:
!pip -q install "transformers>=4.56" "trl>=0.21" "datasets>=3" accelerate
import torch, transformers, trl
print(torch.__version__, transformers.__version__, trl.__version__, torch.cuda.get_device_name(0))

In [ ]:
# Upload ml/data/sft/train.jsonl and val.jsonl
from google.colab import files
uploaded = files.upload()
assert {"train.jsonl", "val.jsonl"} <= set(uploaded), uploaded.keys()

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files={"train": "train.jsonl", "val": "val.jsonl"})

def to_prompt_completion(ex):
    # Train only on the final assistant turn; earlier turns are context.
    msgs = ex["messages"]
    return {"prompt": msgs[:-1], "completion": msgs[-1:]}

ds = ds.map(to_prompt_completion, remove_columns=["messages"])
print(ds)
print(ds["train"][0])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

BASE = "HuggingFaceTB/SmolLM2-135M-Instruct"
bf16 = torch.cuda.is_bf16_supported()
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32)

steps_per_epoch = max(1, len(ds["train"]) // 32)
cfg = SFTConfig(
    output_dir="ckpt",
    num_train_epochs=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, int(0.03 * steps_per_epoch * 2)),
    weight_decay=0.01,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=32,
    max_length=1024,
    completion_only_loss=True,
    bf16=bf16,
    fp16=not bf16,
    eval_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=25,
    report_to="none",
)
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds["train"], eval_dataset=ds["val"],
                     processing_class=tok)
trainer.train()
print(trainer.evaluate())

In [ ]:
# Spot-check on held-out prompts (greedy, like the comparison script).
model = trainer.model.eval()
model.tie_weights()  # SmolLM2 ties lm_head to the embeddings
assert model.lm_head.weight.data_ptr() == model.get_input_embeddings().weight.data_ptr()
for ex in ds["val"].shuffle(seed=0).select(range(8)):
    enc = tok.apply_chat_template(ex["prompt"], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    ids = enc["input_ids"].to(model.device)
    out = model.generate(ids, attention_mask=enc["attention_mask"].to(model.device), max_new_tokens=160,
                         do_sample=False, eos_token_id=[2, 0])
    print("PROMPT:", ex["prompt"][0]["content"].split("\n")[0], "|", ex["prompt"][-1]["content"][:120])
    print("MODEL :", tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip())
    print("TEACH :", ex["completion"][0]["content"])
    print("-" * 80)

In [ ]:
trainer.save_model("astro-135m")
tok.save_pretrained("astro-135m")
!cd astro-135m && ls -la && zip -q -r ../astro-135m.zip .
files.download("astro-135m.zip")